# Attention U-Net FNO — Test-only evaluation

Ez a notebook **nem tanít**, csak:
1. betölti a már betanított `AttentionUNetFNO` modellt,
2. betölti a **test** halmazt,
3. kiszámolja a **loss / Dice / IoU** metrikákat,
4. elmenti az eredményeket,
5. opcionálisan készít néhány predikciós ábrát.

> A pathok már a megadott Google Drive elérési utakra vannak beállítva.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import json
import random
from dataclasses import dataclass
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [ ]:
@dataclass
class Config:
    # reproducibility
    seed_global: int = 42

    # data / eval
    batch_size: int = 16
    num_workers: int = 2
    image_height: int = 256
    image_width: int = 256
    num_classes: int = 2
    modes: int = 4

    # loss
    loss_bce_weight: float = 0.5
    loss_dice_weight: float = 0.5
    loss_smooth: float = 1.0

    # --- user-provided paths ---
    attunet_ckpt: str = '/content/drive/MyDrive/Brain MRI/FNO_seg/AttentionUnet_FNO/models/run1_att_fno_best_model.pt'
    root_path: str = '/content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/test'
    image_dir: str = '/content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/test/images'
    mask_dir: str = '/content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/test/masks'

    # outputs
    output_dir: str = '/content/drive/MyDrive/Brain MRI/FNO_seg/AttentionUnet_FNO/test_eval'
    preds_dir: str = '/content/drive/MyDrive/Brain MRI/FNO_seg/AttentionUnet_FNO/test_eval/predictions'
    results_json: str = '/content/drive/MyDrive/Brain MRI/FNO_seg/AttentionUnet_FNO/test_eval/test_metrics.json'

cfg = Config()

os.makedirs(cfg.output_dir, exist_ok=True)
os.makedirs(cfg.preds_dir, exist_ok=True)

print("Checkpoint exists:", os.path.isfile(cfg.attunet_ckpt), "->", cfg.attunet_ckpt)
print("Image dir exists:", os.path.isdir(cfg.image_dir), "->", cfg.image_dir)
print("Mask dir exists :", os.path.isdir(cfg.mask_dir), "->", cfg.mask_dir)
print("Output dir      :", cfg.output_dir)


Checkpoint exists: True -> /content/drive/MyDrive/Brain MRI/FNO_seg/AttentionUnet_FNO/models/run1_att_fno_best_model.pt
Image dir exists: True -> /content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/test/images
Mask dir exists : True -> /content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/test/masks
Output dir      : /content/drive/MyDrive/Brain MRI/FNO_seg/AttentionUnet_FNO/test_eval


In [ ]:
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed_global)
print("Seed set to:", cfg.seed_global)


Seed set to: 42


In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, image_dir: str, mask_dir: str):
        self.image_dir = image_dir
        self.mask_dir = mask_dir

        self.image_files = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif"))
        ])
        self.mask_files = sorted([
            f for f in os.listdir(mask_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif"))
        ])

        if len(self.image_files) == 0:
            raise ValueError(f"Nincs kép a(z) {image_dir} mappában!")
        if len(self.mask_files) == 0:
            raise ValueError(f"Nincs maszk a(z) {mask_dir} mappában!")
        assert len(self.image_files) == len(self.mask_files), "A képek és maszkok száma nem egyezik!"

        print(f"Találtam {len(self.image_files)} képet és maszkot a teszt halmazon.")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx: int):
        img_path = os.path.join(self.image_dir, self.image_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])

        image = Image.open(img_path).convert("L")
        mask = Image.open(mask_path).convert("L")

        image = image.resize((cfg.image_width, cfg.image_height))
        mask = mask.resize((cfg.image_width, cfg.image_height), Image.NEAREST)

        image = np.array(image, dtype=np.float32) / 255.0
        mask = np.array(mask, dtype=np.int64)
        mask = (mask > 127).astype(np.int64)

        image = np.expand_dims(image, axis=0)

        return torch.from_numpy(image), torch.from_numpy(mask)


In [ ]:
test_dataset = SegmentationDataset(cfg.image_dir, cfg.mask_dir)

test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True
)

images, masks = next(iter(test_loader))
print("Test batch images:", images.shape)
print("Test batch masks :", masks.shape)


Találtam 860 képet és maszkot a teszt halmazon.
Test batch images: torch.Size([16, 1, 256, 256])
Test batch masks : torch.Size([16, 256, 256])


## Modelldefiníció

In [ ]:
@torch.jit.script
def compl_mul2d(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return torch.einsum("bixy, ioxy->boxy", a, b)


class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = 1 / (in_channels * out_channels)

        self.weights1 = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, dtype=torch.cfloat)
        )
        self.weights2 = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, dtype=torch.cfloat)
        )

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfftn(x, dim=[2, 3])

        out_ft = torch.zeros(
            batchsize,
            self.out_channels,
            x.size(-2),
            x.size(-1) // 2 + 1,
            device=x.device,
            dtype=torch.cfloat
        )

        out_ft[:, :, :self.modes1, :self.modes2] = \
            compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = \
            compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)

        x = torch.fft.irfftn(out_ft, s=(x.size(-2), x.size(-1)), dim=[2, 3])
        return x


class FourierLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, modes=4):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=True
        )
        self.conv_fno = SpectralConv2d(in_channels, out_channels, modes, modes)

    def forward(self, x):
        x1 = self.conv(x)
        x2 = self.conv_fno(x)
        return x1 + x2


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, modes=4):
        super().__init__()
        self.net = nn.Sequential(
            FourierLayer(in_ch, out_ch, kernel_size=3, stride=1, padding=1, modes=modes),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            FourierLayer(out_ch, out_ch, kernel_size=3, stride=1, padding=1, modes=modes),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class AttentionBlock(nn.Module):
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(g_ch, inter_ch, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(inter_ch),
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(x_ch, inter_ch, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(inter_ch),
        )
        self.psi = nn.Sequential(
            nn.Conv2d(inter_ch, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


class AttentionUNetFNO(nn.Module):
    def __init__(self, in_channels=1, out_channels=2, modes=4):
        super().__init__()
        # Encoder
        self.down1 = DoubleConv(in_channels, 64, modes=modes)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128, modes=modes)
        self.pool2 = nn.MaxPool2d(2)

        # Bridge
        self.bridge = DoubleConv(128, 256, modes=modes)

        # Attention blokkok
        self.att2 = AttentionBlock(g_ch=128, x_ch=128, inter_ch=64)
        self.att1 = AttentionBlock(g_ch=64, x_ch=64, inter_ch=32)

        # Decoder
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(256, 128, modes=modes)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(128, 64, modes=modes)

        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        b = self.bridge(p2)

        u2 = self.up2(b)
        d2_att = self.att2(g=u2, x=d2)
        u2 = torch.cat([u2, d2_att], dim=1)
        u2 = self.conv2(u2)

        u1 = self.up1(u2)
        d1_att = self.att1(g=u1, x=d1)
        u1 = torch.cat([u1, d1_att], dim=1)
        u1 = self.conv1(u1)

        out = self.out_conv(u1)
        return out


# gyors teszt
model = AttentionUNetFNO(in_channels=1, out_channels=cfg.num_classes, modes=4).to(device)
x_test = torch.randn(4, 1, cfg.image_height, cfg.image_width).to(device)
y_test = model(x_test)
print("Model output shape:", y_test.shape)
print("Total params:", sum(p.numel() for p in model.parameters()))




Model output shape: torch.Size([4, 2, 256, 256])
Total params: 7916840


## Loss és metrikák

In [ ]:
class CombinedBCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5, smooth=1.0):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth
        self.ce_loss = nn.CrossEntropyLoss()

    def dice_loss(self, logits, targets, num_classes=2):
        probs = torch.softmax(logits, dim=1)
        targets_one_hot = torch.nn.functional.one_hot(targets, num_classes=num_classes)
        targets_one_hot = targets_one_hot.permute(0, 3, 1, 2).float()

        dice_scores = []
        for cls in range(num_classes):
            pred_cls = probs[:, cls, :, :]
            target_cls = targets_one_hot[:, cls, :, :]
            intersection = (pred_cls * target_cls).sum()
            union = pred_cls.sum() + target_cls.sum()
            dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
            dice_scores.append(dice)
        mean_dice = sum(dice_scores) / len(dice_scores)
        return 1.0 - mean_dice

    def forward(self, logits, targets):
        ce_loss = self.ce_loss(logits, targets)
        dice_loss = self.dice_loss(logits, targets, num_classes=logits.size(1))
        return self.bce_weight * ce_loss + self.dice_weight * dice_loss


def dice_score(logits, targets, num_classes=2):
    preds = torch.argmax(logits, dim=1)
    batch_size = preds.size(0)
    sample_dice_scores = []
    for i in range(batch_size):
        dice_scores = []
        for cls in range(num_classes):
            pred_mask = (preds[i] == cls).float()
            target_mask = (targets[i] == cls).float()
            intersection = (pred_mask * target_mask).sum()
            dice = (2.0 * intersection) / (pred_mask.sum() + target_mask.sum() + 1e-8)
            dice_scores.append(dice.item())
        sample_dice_scores.append(sum(dice_scores) / len(dice_scores))
    return sample_dice_scores


def compute_iou(logits, targets, num_classes=2):
    preds = torch.argmax(logits, dim=1)
    batch_size = preds.size(0)
    sample_iou_scores = []
    for i in range(batch_size):
        ious = []
        for cls in range(num_classes):
            pred_inds = (preds[i] == cls)
            target_inds = (targets[i] == cls)
            intersection = (pred_inds & target_inds).sum().item()
            union = (pred_inds | target_inds).sum().item()
            if union == 0:
                continue
            ious.append(intersection / union)
        if len(ious) == 0:
            sample_iou_scores.append(0.0)
        else:
            sample_iou_scores.append(sum(ious) / len(ious))
    return sample_iou_scores


## Kiértékelés a test halmazon

In [ ]:
criterion = CombinedBCEDiceLoss(
    bce_weight=cfg.loss_bce_weight,
    dice_weight=cfg.loss_dice_weight,
    smooth=cfg.loss_smooth,
)

def load_model_from_checkpoint(checkpoint_path: str):
    model = AttentionUNetFNO(in_channels=1, out_channels=cfg.num_classes, modes=cfg.modes).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict)
    model.eval()
    return model

@torch.no_grad()
def evaluate_model(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    total_samples = 0
    all_dice = []
    all_iou = []

    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, masks)

        running_loss += loss.item() * images.size(0)
        total_samples += images.size(0)

        batch_dice = dice_score(outputs, masks, num_classes=cfg.num_classes)
        batch_iou = compute_iou(outputs, masks, num_classes=cfg.num_classes)

        all_dice.extend(batch_dice)
        all_iou.extend(batch_iou)

    metrics = {
        "test_loss": float(running_loss / max(total_samples, 1)),
        "test_dice": float(np.mean(all_dice)) if len(all_dice) > 0 else 0.0,
        "test_iou": float(np.mean(all_iou)) if len(all_iou) > 0 else 0.0,
        "num_samples": int(total_samples),
    }
    return metrics

best_model = load_model_from_checkpoint(cfg.attunet_ckpt)
test_metrics = evaluate_model(best_model, test_loader, criterion)

print("=== TEST RESULTS ===")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

with open(cfg.results_json, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2, ensure_ascii=False)

print("\nEredmények elmentve ide:")
print(cfg.results_json)


=== TEST RESULTS ===
test_loss: 0.052567520651013354
test_dice: 0.923386210028277
test_iou: 0.8848206950838384
num_samples: 860

Eredmények elmentve ide:
/content/drive/MyDrive/Brain MRI/FNO_seg/AttentionUnet_FNO/test_eval/test_metrics.json


## Opcionális vizualizáció

In [ ]:
def visualize_and_save_predictions(model, loader, num_samples=4, save_path=None):
    model.eval()
    images, masks = next(iter(loader))
    images = images.to(device)
    masks = masks.to(device)

    with torch.no_grad():
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

    images = images.cpu().numpy()
    masks = masks.cpu().numpy()
    preds = preds.cpu().numpy()

    num_samples = min(num_samples, len(images))
    plt.figure(figsize=(12, num_samples * 3))

    for i in range(num_samples):
        plt.subplot(num_samples, 3, i * 3 + 1)
        plt.imshow(images[i][0], cmap="gray")
        plt.title("Input Image")
        plt.axis("off")

        plt.subplot(num_samples, 3, i * 3 + 2)
        plt.imshow(masks[i], cmap="gray")
        plt.title("Ground Truth")
        plt.axis("off")

        plt.subplot(num_samples, 3, i * 3 + 3)
        plt.imshow(preds[i], cmap="gray")
        plt.title("Model Prediction")
        plt.axis("off")

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight")
        plt.close()
        print("Predikciós ábra mentve:", save_path)
    else:
        plt.show()

preds_path = os.path.join(cfg.preds_dir, "att_unet_fno_test_predictions.png")
visualize_and_save_predictions(best_model, test_loader, num_samples=4, save_path=preds_path)


Predikciós ábra mentve: /content/drive/MyDrive/Brain MRI/FNO_seg/AttentionUnet_FNO/test_eval/predictions/att_unet_fno_test_predictions.png


## Futtatási sorrend
Fentről lefelé futtasd a cellákat.  
A notebook **nem** tartalmaz tréninget, csak tesztkiértékelést.
